# NEOARM calibration, one section at a time

Calibrate the **sample-to-detector section first**, then use its measured response to calibrate the **source-to-sample section**. This notebook is the cleaned experiment entry point: it audits the recorded free-lens controls and defines the next measurements. It does not run a new inversion or the historical multistart jobs.

The detailed rationale and acceptance criteria are in [the recipe](README.md). Previous source code and results are indexed in [the research checkpoint](archive/README.md).

## 1. The split is supported by the paper

On PDF p. 6, Pfeifer, Lupini, and Hoglund start the projector rays at the sample plane to prevent condenser errors from propagating into that model. Their condenser calibration uses its own beam and aperture measurements. Their focusing and rotation measurements constrain complementary lens properties, while Table II still reports 6.3–28% camera-length errors; their results support useful predictive calibration, not unique sub-percent recovery of every internal parameter.

Read the [supplied PDF](paper/2607.29411v1.pdf) or [arXiv version](https://arxiv.org/abs/2607.29411).

```text
source → CL1 → CL3 → C-mini → objective prefield
                             ↓
                sample illumination + specimen
                             ↓
objective postfield → IL1 → IL2 → IL3 → PL1 → detector
```

Both sections use the same specimen plane, voltage, coordinate convention, and objective-field hypothesis. Objective controls remain fixed; that does not make its physical field exactly known.

In [ ]:
from pathlib import Path
import sys
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.ticker import NullFormatter
from IPython.display import Markdown, display

HERE = Path.cwd().resolve()
EXAMPLES = HERE if (HERE / 'dac_lens_inversion.ipynb').is_file() else HERE / 'examples/lens_inversion'
assert (EXAMPLES / 'dac_lens_inversion.ipynb').is_file(), 'Run from the repository or this notebook directory.'
ROOT = EXAMPLES.parents[1]
sys.path.insert(0, str(EXAMPLES))
from neoarm_flc import read_flc_table, PROJECTOR_LENSES, CONTROL_NAMES

## 2. Begin with the recorded controls

The reader below extracts only the literal `RAW_DAC_HEX` dictionary from the existing DAC notebook. It executes no notebook code and does not use stored fitting outputs. The magnifications are **nominal labels**, and the DAC words are **not measured lens currents or approved scan commands**.

In [ ]:
table = read_flc_table(EXAMPLES / 'dac_lens_inversion.ipynb')
magnifications = table['nominal_magnification']
header = '| Nominal magnification | ' + ' | '.join(CONTROL_NAMES) + ' |'
separator = '| ---: | ' + ' | '.join(['---:'] * len(CONTROL_NAMES)) + ' |'
rows = ['| ' + str(mag) + ' | ' + ' | '.join(str(table[name][i]) for name in CONTROL_NAMES) + ' |'
        for i, mag in enumerate(magnifications)]
display(Markdown('\n'.join([header, separator, *rows])))

In [ ]:
colors = ('#0072B2', '#D55E00', '#009E73', '#6A51A3')
with plt.rc_context({'font.size': 10, 'axes.spines.top': False, 'axes.spines.right': False}):
    fig, axes = plt.subplots(2, 2, figsize=(9, 5.7), sharex=True, layout='constrained')
    for ax, name, color in zip(axes.flat, PROJECTOR_LENSES, colors):
        ax.plot(magnifications / 1000, table[name], 'o-', color=color, lw=1.6, ms=4)
        ax.set(title=name, xscale='log', ylabel='Raw DAC word')
        ax.set_xticks([40, 80, 150, 300, 600], ['40', '80', '150', '300', '600'])
        ax.xaxis.set_minor_formatter(NullFormatter())
        ax.ticklabel_format(axis='y', style='plain', useOffset=False)
        ax.grid(alpha=.18)
        if np.ptp(table[name]) == 0:
            ax.set_ylim(table[name][0] - 1000, table[name][0] + 1000)
            ax.text(.04, .84, 'No excitation variation in this table', transform=ax.transAxes, fontsize=9)
    for ax in axes[-1]:
        ax.set_xlabel('Nominal magnification (thousands ×)')
    fig.suptitle('Recorded projector controls, 40k–600k')
plt.show()

In [ ]:
controls = np.column_stack([table[name] for name in PROJECTOR_LENSES])
centered = controls - controls.mean(axis=0)
constant = [name for name in CONTROL_NAMES if np.ptp(table[name]) == 0]
print('Recorded settings:', len(magnifications))
print('Constant controls:', ', '.join(constant))
print('Centered control-table rank:', np.linalg.matrix_rank(centered), '(not a lens-identifiability rank)')

OLf, OLc, and PL1 are constant across these 13 settings. IL2 and IL3 largely compensate as magnification changes, while IL1 changes over a smaller range. The table supplies a useful operating trajectory; it does not supply exact focus, calibrated magnification, or measured rotation.

A deliberate PL1 perturbation is needed to measure its command response. The table has no CL1, CL3, or C-mini entries, so the condenser needs its own acquisition.

In [ ]:
from temgym_core.plotting import export_ray_figure

outputs = export_ray_figure(fig, ROOT / 'examples/generated/neoarm_inversion/section_recipe/flc_controls',
    metadata={'title': 'Recorded NEOARM free-lens controls',
              'source': 'examples/lens_inversion/dac_lens_inversion.ipynb',
              'scope': 'Raw control audit; nominal magnifications; no optical fit'})
print('Exported SVG, PDF, 300-dpi PNG, and metadata to', outputs['svg'].parent)

## 3. Fit a section with a declared input

For mechanical sample-plane coordinates $\mathbf r_s=(x,y,\alpha_x,\alpha_y)$, the detector position is

$$\mathbf x_d=A\mathbf x_s+B\boldsymbol\alpha_s+\mathbf b.$$

At an image plane, the certified specimen constrains $A$ through scale, rotation, and distortion. Calibrated sample-plane tilt responses or indexed diffraction measurements add angular information. A focused image alone cannot identify every internal lens parameter.

For each effective finite lens, begin with $q_i(D_i)=a_iD_i$, where $q_i$ is integrated Larmor excitation and $D_i$ is the raw control. Fit the finite rotating matrix already in the library. Its rotation is $-q_i$, and its weak-lens power is approximately $1/f_i=q_i^2/L_i$. Current readback is unnecessary because the gain absorbs the control-to-field conversion; proportionality and zero offset still require measurement checks.

Start with approximate geometry held fixed, then release only lengths and positions that the measured responses can distinguish. Constant objective controls require a consistent postfield model or a calibrated effective map, not an arbitrary independently fitted objective half.

## 4. The first downstream experiment

1. Record the same registered multiscale specimen near 40k and 80k, including repeat frames to measure drift, dose variation, and image-measurement precision.
2. Apply small positive and negative perturbations to IL1, IL2, IL3, and PL1 within operator-approved ranges, leaving the condenser and objective controls fixed.
3. Extract specimen scale, rotation, and distortion, plus calibrated position/angle responses where available. Use asymmetric texture to resolve periodic-pattern orientation ambiguities.
4. Add a small paired lens scan where candidate models predict different crossover or conjugacy behaviour. Fit the whole downstream section to the observations; an isolated diameter minimum is insufficient.
5. Fit excitation gains first, then selected geometry. Use continuous scans and coarse branch searches to initialize distinct crossover regimes.
6. Predict mixed-lens settings withheld before model selection. Compare fresh scale, rotation, centroid, and indexed diffraction vectors with measured uncertainties and practical tolerances.

There is no fixed 230-frame requirement at this stage. The first measurements determine noise, usable wobble ranges, and which additional states can separate the surviving models.

## 5. The condenser uses separate measurements

A calibrated downstream section can observe the specimen-plane beam through several analyser states, allowing size, angular spread, and position–angle correlations to be distinguished. The analyser uncertainties remain shared across the condenser fit.

Vary CL1, CL3, and C-mini with a reproducible astigmatic illumination state, fitting the entrance covariance as a nuisance. An observed ellipse angle is not automatically the sum of lens rotations: the unknown stigmator location and general source covariance affect that interpretation. No calibrated upstream beam shift or tilt is assumed.

Add unnormalised aperture-transmission measurements and a source-dose reference. A stationary aperture can provide a lens-sweep intensity curve; translation adds information if reproducible. Its plane, diameter, and ordering relative to the lenses still need confirmation for NEOARM. The paper's C1–VOA–C2–C3 arrangement is not established for our column.

Detector clipping and specimen absorption must not be mistaken for aperture loss. In particular, the small specimen perforation in the synthetic atlas is not automatically a total-flux collector.

## 6. Images validate the measurements; joint fitting is optional

The existing cubic specimen and GPU Gaussian renderer remain available to generate noisy detector images at a chosen magnification and verify observable extraction. Millions of beamlets are allowed, with `gpu_kernel_required` for forward rendering. The initial lens fit uses extracted observables and their covariance, avoiding a full specimen render at every optimizer step.

Before interpreting a lens fit, check sample interpolation, Gaussian expansion, detector integration, and physical model accuracy separately. Some historical condenser states passed first-order fitting but failed a nonparaxial sensitivity check; those states require lower angles or improved physical modelling.

The two sections can remain separately calibrated. An optional final refinement combines their original data once, sharing the objective and detector parameters; it must not count the projector calibration both as original images and as a posterior prior.

Ray diagrams for subsequent fitted states use the retained plotting system: true-z overviews and section zooms, calibrated height/angle inputs, the Gaussian envelope $w(z)=\sqrt{x_h(z)^2+x_\alpha(z)^2}$, and identical comparison extents.

## Next bounded result

Fit a fixed-geometry downstream model with a simple excitation law to the recorded operating trajectory and measured calibration responses, then predict withheld settings. The code above establishes the control-table input only; it does not yet perform that fit.

The acceptance targets and the distinction between predictive calibration and physical-parameter identification are in [the recipe](README.md). Historical experiments remain recoverable from git checkpoint `3d51e0e`; [the archive index](archive/README.md) records their evidence and limitations.